# Phase 10: Hyperparameter optimization

Tune regressors with Optuna and GroupKFold so all cycles from a given engine remain in one fold. The optimized CV metric is selected before refitting the best estimator on the full training feature matrix.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from src.features.feature_engineering import build_feature_matrix
from src.features.target_engineering import prepare_fd_dataset
from src.models.hyperparameter_optimization import optimize_hyperparameters

raw_dir = Path('../data/raw/CMAPSSData')
fd_data = prepare_fd_dataset('FD001', raw_dir=raw_dir)
feature_matrix = build_feature_matrix(fd_data['train'], target_column='rul_uncapped', windows=(3, 5, 10))

optimization = optimize_hyperparameters(
    feature_matrix,
    model_name='random_forest',
    target_column='rul_uncapped',
    group_column='engine_id',
    metric='rmse',
    n_splits=3,
    n_trials=30,
    random_state=42,
)
Path('../reports').mkdir(parents=True, exist_ok=True)
optimization.trials.to_csv('../reports/phase10_optimization_trials.csv', index=False)
print(f'Best engine-grouped CV RMSE: {optimization.best_value:.3f}')
print(f'Best parameters: {optimization.best_params}')
optimization.trials.sort_values('value').head()